In [ ]:
#eeg-5band-freq

In [1]:
#نرمالایزیشن
import numpy as np, os

OUTPUT_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"
sid = "VP001"

eeg = np.load(os.path.join(OUTPUT_PATH, sid, f"{sid}_eeg_windows.npy"))
labels = np.load(os.path.join(OUTPUT_PATH, sid, f"{sid}_labels.npy"))

# آیا اختلاف معناداری بین کلاس‌ها وجود داره؟
for cls in [0, 1, 2]:
    cls_data = eeg[labels == cls]
    print(f"Class {cls}: mean={cls_data.mean():.4f}, std={cls_data.std():.4f}, "
          f"n={len(cls_data)}")

# اختلاف بین کلاس‌ها (باید معنادار باشه)
diff_01 = np.abs(eeg[labels==0].mean() - eeg[labels==1].mean())
diff_02 = np.abs(eeg[labels==0].mean() - eeg[labels==2].mean())
diff_12 = np.abs(eeg[labels==1].mean() - eeg[labels==2].mean())
print(f"\nMean difference 0vs1: {diff_01:.6f}")
print(f"Mean difference 0vs2: {diff_02:.6f}")
print(f"Mean difference 1vs2: {diff_12:.6f}")
print(f"\nOverall std: {eeg.std():.4f}")
print(f"Signal-to-noise ratio (diff/std): {diff_12/eeg.std():.4f}")
print("(اگه این عدد خیلی کوچیکه، نرمالایزیشن مشکل داره)")

Class 0: mean=-0.0003, std=0.7636, n=327
Class 1: mean=-0.0001, std=0.7444, n=338
Class 2: mean=-0.0002, std=0.7613, n=326

Mean difference 0vs1: 0.000231
Mean difference 0vs2: 0.000162
Mean difference 1vs2: 0.000069

Overall std: 0.7564
Signal-to-noise ratio (diff/std): 0.0001
(اگه این عدد خیلی کوچیکه، نرمالایزیشن مشکل داره)


In [2]:
#Receptive Field test
import torch
import torch.nn as nn

print("="*60)
print("TEST 2: EEG Branch Temporal Receptive Field")
print("="*60)

# محاسبه‌ی دقیق receptive field مدل شما
# Conv1: kernel=(12,4,4), stride=(6,2,2)
# Conv2: kernel=(6,2,2), stride=(2,2,2)

def calc_receptive_field(layers):
    """layers: list of (kernel, stride)"""
    rf = 1
    total_stride = 1
    for kernel, stride in layers:
        rf = rf + (kernel - 1) * total_stride
        total_stride *= stride
    return rf, total_stride

# فقط temporal dimension (dim 0 در D,H,W = time با size 600)
layers_temporal = [
    (12, 6),   # Conv1 temporal
    (6, 2),    # Conv2 temporal
]

rf, stride = calc_receptive_field(layers_temporal)
rf_ms = rf * (1000 / 200)  # 200Hz sampling rate
output_size = (600 - 12) // 6 + 1
output_size2 = (output_size - 6) // 2 + 1

print(f"\nEEG temporal dimension: 600 samples = 3000ms (3 seconds)")
print(f"\nConv1: kernel=12, stride=6")
print(f"  Output size: {output_size} time steps")
print(f"  Each output sees: 12 samples = {12*1000/200:.0f}ms of signal")
print(f"\nConv2: kernel=6, stride=2")
print(f"  Output size: {output_size2} time steps")
print(f"  Each output sees: {rf} samples = {rf_ms:.0f}ms of signal")

print(f"\nTotal temporal receptive field: {rf} samples = {rf_ms:.0f}ms")
print(f"Final feature map: {output_size2} time steps, then averaged → 1 vector")

print(f"\n{'='*60}")
print(f"INTERPRETATION:")
print(f"  P300 (key cognitive ERP): peaks at 300-600ms, lasts ~300ms")
print(f"  Alpha power changes: need 500-2000ms window to detect reliably")
print(f"  Your receptive field: {rf_ms:.0f}ms")

if rf_ms < 500:
    print(f"  ❌ Receptive field ({rf_ms:.0f}ms) < 500ms")
    print(f"     مدل نمی‌تونه P300 کامل یا alpha power changes رو ببینه")
    print(f"     → پیشنهاد: kernel_time در Conv1 رو از 12 به 24 برسونید")
elif rf_ms < 1000:
    print(f"  ⚠️  Receptive field ({rf_ms:.0f}ms) کافیه ولی tight هست")
    print(f"     P300 رو می‌بینه ولی alpha/theta power changes رو ضعیف می‌بینه")
else:
    print(f"  ✅ Receptive field کافیه")

# تست عملی: آیا mismatch بین conv output size و feature map هست؟
print(f"\n{'='*60}")
print(f"PRACTICAL CHECK:")
print(f"  After eeg_conv1: (batch, 16, {output_size}, 8, 8)")
print(f"  After eeg_conv2: (batch, 32, {output_size2}, 4, 4)")
print(f"  After mean(dim=2): (batch, 32, 4, 4)")
print(f"  After flatten: (batch, {32*4*4})")
print(f"  eeg_fc input: 32×4×4 = {32*4*4} ✅" if output_size2 > 0 else "❌ Output size is 0!")

TEST 2: EEG Branch Temporal Receptive Field

EEG temporal dimension: 600 samples = 3000ms (3 seconds)

Conv1: kernel=12, stride=6
  Output size: 99 time steps
  Each output sees: 12 samples = 60ms of signal

Conv2: kernel=6, stride=2
  Output size: 47 time steps
  Each output sees: 42 samples = 210ms of signal

Total temporal receptive field: 42 samples = 210ms
Final feature map: 47 time steps, then averaged → 1 vector

INTERPRETATION:
  P300 (key cognitive ERP): peaks at 300-600ms, lasts ~300ms
  Alpha power changes: need 500-2000ms window to detect reliably
  Your receptive field: 210ms
  ❌ Receptive field (210ms) < 500ms
     مدل نمی‌تونه P300 کامل یا alpha power changes رو ببینه
     → پیشنهاد: kernel_time در Conv1 رو از 12 به 24 برسونید

PRACTICAL CHECK:
  After eeg_conv1: (batch, 16, 99, 8, 8)
  After eeg_conv2: (batch, 32, 47, 4, 4)
  After mean(dim=2): (batch, 32, 4, 4)
  After flatten: (batch, 512)
  eeg_fc input: 32×4×4 = 512 ✅


In [3]:
#
import torch

print("="*60)
print("TEST 2: EEG Branch Temporal Receptive Field")
print("="*60)

def calc_receptive_field(layers):
    rf = 1
    total_stride = 1
    for kernel, stride in layers:
        rf = rf + (kernel - 1) * total_stride
        total_stride *= stride
    return rf, total_stride

layers_temporal = [(12, 6), (6, 2)]
rf, stride = calc_receptive_field(layers_temporal)
rf_ms = rf * (1000 / 200)

output_size  = (600 - 12) // 6 + 1
output_size2 = (output_size - 6) // 2 + 1

print(f"\nConv1: kernel=12, stride=6 → output: {output_size} time steps")
print(f"Conv2: kernel=6,  stride=2 → output: {output_size2} time steps")
print(f"\nTotal temporal receptive field: {rf} samples = {rf_ms:.0f}ms")

print(f"\nINTERPRETATION:")
print(f"  P300 (مهم‌ترین ERP شناختی): peak در 300-600ms")
print(f"  Alpha/theta power: نیاز به 500-2000ms")
print(f"  Receptive field شما: {rf_ms:.0f}ms")

if rf_ms < 500:
    print(f"  ❌ خیلی کوچیکه — مدل P300 کامل رو نمی‌بینه")
    print(f"  راه‌حل: Conv1 kernel_time رو از 12 به 24 برسونید")
elif rf_ms < 1000:
    print(f"  ⚠️  قابل قبوله ولی tight هست")
else:
    print(f"  ✅ کافیه")

print(f"\nPRACTICAL CHECK:")
print(f"  After Conv1: (batch, 16, {output_size}, 8, 8)")
print(f"  After Conv2: (batch, 32, {output_size2}, 4, 4)")
print(f"  eeg_fc input: 32×4×4 = {32*4*4}")


TEST 2: EEG Branch Temporal Receptive Field

Conv1: kernel=12, stride=6 → output: 99 time steps
Conv2: kernel=6,  stride=2 → output: 47 time steps

Total temporal receptive field: 42 samples = 210ms

INTERPRETATION:
  P300 (مهم‌ترین ERP شناختی): peak در 300-600ms
  Alpha/theta power: نیاز به 500-2000ms
  Receptive field شما: 210ms
  ❌ خیلی کوچیکه — مدل P300 کامل رو نمی‌بینه
  راه‌حل: Conv1 kernel_time رو از 12 به 24 برسونید

PRACTICAL CHECK:
  After Conv1: (batch, 16, 99, 8, 8)
  After Conv2: (batch, 32, 47, 4, 4)
  eeg_fc input: 32×4×4 = 512
